In [ ]:
import json as js
import sys
import pandas as pd
import altair as alt
from harpy.report.theme import palette


In [6]:
class Molecule():
    def __init__(self):
        self.A = 0.0
        self.B = 0.0
        self.C = 0.0
    def update(self, segment, frac: float):
        setattr(self, segment, round(frac,5))
    def __str__(self) -> str:
        return f"{self.A}\t{self.B}\t{self.C}"

def process_json(json):
    pheniqs = js.load(open(json, 'r'))
    d = {}
    total = pheniqs['incoming']['count']
    for i,j in enumerate(["A","B","C"],1):
        #d[f'{j}-unclassified'] = round(1- pheniqs['molecular'][i]['classified fraction'], 5)
        for k in pheniqs['molecular'][i]['classified']:
            barcode = k['barcode'][0]
            if barcode not in d:
                d[barcode] = Molecule()
            d[barcode].update(j, k['pooled fraction'])
    return total,d


In [7]:
json = "/home/pdimens/Documents/harpy/Preprocess/logs/extract/gih_001.json"
total, dd = process_json(json)
df = pd.DataFrame.from_dict(
    {bc: vars(mol) for bc, mol in dd.items()},
    orient='index'
).rename_axis('barcode').reset_index()
df.columns = ['barcode', "sample1_A","sample1_B","sample1_C"]
df


,barcode,sample1_A,sample1_B,sample1_C
0,TTACCAGGTCCG,0.00967,0.00967,0.00866
1,AGACCATCGCCG,0.01007,0.01209,0.01451
2,CACTTACCGTGA,0.01148,0.00846,0.00685
3,GAGTCGGACCGC,0.00725,0.00987,0.00987
4,TCAGCGATGGTT,0.00927,0.00967,0.01007
...,...,...,...,...
99,CGGTGGTTATCA,0.00987,0.01330,0.00806
100,ACGTGCGGTCAT,0.01048,0.00887,0.01269
101,TGTGGCCGGCCT,0.00000,0.00000,0.00000
102,TTCCCGTCGCGT,0.00000,0.00000,0.00000


In [69]:
def plot_from_wide(df: pd.DataFrame) -> alt.Chart:
    samples = list(dict.fromkeys(col.rsplit('_', 1)[0] for col in df.columns[1:] if '_' in col))
    segments = ['A', 'B', 'C']
    all_cols = [f'{s}_{seg}' for s in samples for seg in segments]

    sample_select = alt.binding_select(options=samples, name='Sample: ')
    segment_select = alt.binding_select(options=segments, name='Segment: ')
    sample_param = alt.param(name='sample', bind=sample_select, value=samples[0])
    segment_param = alt.param(name='segment', bind=segment_select, value='A')

    base = alt.Chart(df).transform_fold(
        all_cols, as_=['col', 'value']
    ).transform_calculate(
        sample="split(datum.col, '_')[0]",
        segment="split(datum.col, '_')[1]"
    ).transform_filter(
        'datum.sample == sample && datum.segment == segment'
    )

    #x = alt.X('barcode:N', sort=alt.EncodingSortField(field='value', order='descending'))#,axis=alt.Axis(labelAngle=-45))

    chart = base.mark_point().encode(
        x=alt.X('barcode:N', sort=alt.EncodingSortField(field='value', order='descending')),
        y=alt.Y('value:Q', title='Fraction'),
        tooltip=[
            alt.Tooltip('barcode:N', title = "Barcode"),
            alt.Tooltip('segment:N', title = "Segment"),
            alt.Tooltip('value:Q', title = "Fraction")
        ]
    )

    return chart.add_params(
        sample_param, segment_param
    ).properties(title='Barcode Fractions')


In [70]:
plot_from_wide(df)


alt.Chart(...)